In [ ]:
# Loading model and basic timing
import torch
from transformers import GPT2Model, GPT2Config
import time

print(f"GPU : {torch.cuda.get_device_name(0)}\n")

# Loading GPT-2 
config = GPT2Model(GPT2Config(
    n_layer=12, # 12 transformer blocks stacked
    n_head=12, # 12 attention heads per block
    n_embd=768 # each token is a vector of 768 numbers
)).cuda().eval()

model = config
params = sum(p.numel() for p in model.parameters())
print(f"Model loaded : {params/1e6:.1f}M parameters")
print(f"Model size :   {params*4/1024**2:.0f} MB in FP32\n")
# To get MB we have to divide 1024 twice. so ** 2

GPU : Tesla T4  
Model loaded : 124.4M parameters 
Model size :   475 MB in FP32

In [ ]:
# Creating fake input and run
batch_size = 8    # 8 sentences at once
seq_len    = 128  # each sentence is 128 tokens
vocab_size = 50257  # GPT-2 has 50,257 possible tokens

# Random token IDs — fake input representing 8 sentences of 128 tokens each
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len)).cuda()
print(f"Input shape : {input_ids.shape}")
print(f"  - {batch_size} sentences, each {seq_len} tokens long\n")

# Warming up — first run is always slow due to CUDA initialisation
# Running 5 times to warm up then we measure
print("Warming up start")
with torch.no_grad():
    for _ in range(5):
        _ = model(input_ids)
torch.cuda.synchronize()
print("Warm up done.\n")

# Now measure average time over 20 runs
runs = 20
times = []

with torch.no_grad():
    for _ in range(runs):
        torch.cuda.synchronize()  # wait for GPU to finish any old tasks
        t0 = time.perf_counter()  # start timer

        output = model(input_ids)  # run forward pass

        torch.cuda.synchronize()  # wait for GPU to finish the above task
        t1 = time.perf_counter()  # stop timer

        times.append((t1 - t0) * 1000)  # convert to milliseconds

avg = sum(times) / len(times)
print(f"Average forward pass time : {avg:.2f} ms")
print(f"Throughput : {batch_size * seq_len / avg * 1000:.0f} tokens/second")

Input shape : torch.Size([8, 128])- 8 sentences, each 128 tokens long  Warming up start
Warm up done.  
Average forward pass time : 57.33 ms Throughput : 17862 tokens/second

In [ ]:
# Profile the forward pass
from torch.profiler import profile, record_function, ProfilerActivity

print("Profiling GPT-2 forward pass")
print("This runs the model 10 times and records every operation.\n")

with torch.no_grad():
    with profile(
        # Record both CPU and GPU activity just to know we can do it
        # But we will consider only GPU tasks
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=True,   # record tensor shapes
        with_flops=True       # estimate FLOPs
    ) as prof:
        # Run 10 times inside the profiler
        for _ in range(10):
            output = model(input_ids)
        torch.cuda.synchronize()

# Show top 15 operations sorted by GPU time
print("Top 15 GPU operations (sorted by time on GPU) :\n")
print(prof.key_averages().table(
    sort_by="self_cuda_time_total", # sorting happens here
    row_limit=15
))

Operation                    GPU Time    % Total   Calls
─────────────────────────────────────────────────────────
aten::addmm (Linear/GEMM)   467.501ms   73.54%    480
volta_sgemm_128x64_nn        312.908ms   49.22%    360   - GEMM kernel
volta_sgemm_128x128_nn       154.105ms   24.24%    120   - GEMM kernel
aten::mul                     56.719ms    8.92%    480
aten::add                     40.573ms    6.38%    490
_efficient_attention_forward  25.886ms    4.07%    120   - FlashAttention
aten::tanh                    12.807ms    2.01%    120   - GELU uses tanh
aten::pow                     12.801ms    2.01%    120   - GELU uses x³

In [ ]:
# Checking if time scale linearly or quadratically?
# If FFN dominates -> doubling seq_len should double time (linear)
# If attention dominates -> doubling seq_len should 4x time (quadratic)

print("Sequence length scaling test :\n")
print(f"{'SeqLen':<10} {'Time (ms)':<12} {'vs prev':<10}")
print("-" * 32)

prev_time = None
with torch.no_grad():
    for sl in [64, 128, 256, 512]:
        ids = torch.randint(0, vocab_size, (batch_size, sl)).cuda()

        # Warm up again 3 times then later we estimate
        for _ in range(3):
            _ = model(ids)
        torch.cuda.synchronize()

        # Measure npw
        t_list = []
        for _ in range(10):
            torch.cuda.synchronize() # just to tackel any old tasks
            t0 = time.perf_counter()
            _ = model(ids)
            torch.cuda.synchronize()
            t1 = time.perf_counter()
            t_list.append((t1-t0)*1000)

        avg_t = sum(t_list)/len(t_list)
        ratio = f"{avg_t/prev_time:.2f}x" if prev_time else "baseline"
        print(f"{sl:<10} {avg_t:<12.2f} {ratio:<10}")
        prev_time = avg_t

print("\nIf ratio is 2x then linear scaling -> FFN dominates")
print("If ratio is 4x then quadratic scaling -> attention dominates")
# Becuase Attention scales at 4x and FFN scales at 2x

SeqLen    Time (ms)    Ratio
64        32.00        baseline
128       59.14        1.85x
256       122.11       2.06x
512       241.87       1.98x

Every time seq_len doubles, time roughly doubles — around 2x.

FFN completely dominates at these sequence lengths.

FFN is O(N) — each token processed independently through the weight matrix. Double the tokens - double the work - double the time. Exactly what we measure.

If attention dominated we would see 4x when seq_len doubles because attention is O(N²). we see 2x — so attention is not the bottleneck at seq_len 64-512.

This matches our profiler output perfectly :

Attention : only 4.07% at seq_len=128
GEMM :      73.54% completely dominant



